In [1]:
import os
import sys
import pyspark

os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 2g pyspark-shell"

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITO5202 Assessment 1")
    .master("local[2]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("ERROR")

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/New_York"
)

print("Spark version:", spark.version)
print("Master:", sc.master)
print("Default parallelism:", sc.defaultParallelism)
print("Driver memory:", sc.getConf().get("spark.driver.memory"))
print("Spark UI:", sc.uiWebUrl)

Spark version: 4.1.1
Master: local[2]
Default parallelism: 2
Driver memory: 2g
Spark UI: http://b60ecd721b18:4040


# Part A: Analytical Query Design and Implementation

## 1. Dataset Loading and Initial Exploration

In [3]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampType,
    IntegerType
)

trip_schema_jan = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

In [4]:
zone_schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

In [5]:
zone_df = (
    spark.read
    .option("header", True)
    .schema(zone_schema)
    .csv("data/taxi_zone_lookup.csv")
)

zone_df.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


In [6]:
zone_df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [7]:
jan_check_df = spark.read.parquet(
    "data/yellow_tripdata_2023-01.parquet"
)

jan_check_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampNTZType
)

jan_source_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

In [9]:
jan_df = (
    spark.read
    .schema(jan_source_schema)
    .parquet("data/yellow_tripdata_2023-01.parquet")
)

jan_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [10]:
from pyspark.sql.functions import col

jan_standardised_df = jan_df.select(
    col("VendorID").cast("long").alias("VendorID"),
    col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
    col("passenger_count").cast("long").alias("passenger_count"),
    col("trip_distance").cast("double").alias("trip_distance"),
    col("RatecodeID").cast("long").alias("RatecodeID"),
    col("store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
    col("PULocationID").cast("long").alias("PULocationID"),
    col("DOLocationID").cast("long").alias("DOLocationID"),
    col("payment_type").cast("long").alias("payment_type"),
    col("fare_amount").cast("double").alias("fare_amount"),
    col("extra").cast("double").alias("extra"),
    col("mta_tax").cast("double").alias("mta_tax"),
    col("tip_amount").cast("double").alias("tip_amount"),
    col("tolls_amount").cast("double").alias("tolls_amount"),
    col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
    col("total_amount").cast("double").alias("total_amount"),
    col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
    col("airport_fee").cast("double").alias("airport_fee")
)

In [11]:
jan_standardised_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [12]:
jan_standardised_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |1              |0.97         |1         |N                 |161         |141         |2           |9.3        |1.0  |0.5    |0.0      

In [13]:
feb_check_df = spark.read.parquet(
    "data/yellow_tripdata_2023-02.parquet"
)

feb_check_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [14]:
for month in range(1, 13):
    path = f"data/yellow_tripdata_2023-{month:02d}.parquet"

    print(f"\nMonth: {month:02d}")
    spark.read.parquet(path).printSchema()


Month: 01
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)


Month: 02
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-

In [15]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    DoubleType,
    StringType,
    TimestampNTZType
)

later_source_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
])

In [16]:
from pyspark.sql.functions import col

def standardise_trip_df(df, airport_column):
    return df.select(
        col("VendorID").cast("long").alias("VendorID"),
        col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
        col("passenger_count").cast("long").alias("passenger_count"),
        col("trip_distance").cast("double").alias("trip_distance"),
        col("RatecodeID").cast("long").alias("RatecodeID"),
        col("store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
        col("PULocationID").cast("long").alias("PULocationID"),
        col("DOLocationID").cast("long").alias("DOLocationID"),
        col("payment_type").cast("long").alias("payment_type"),
        col("fare_amount").cast("double").alias("fare_amount"),
        col("extra").cast("double").alias("extra"),
        col("mta_tax").cast("double").alias("mta_tax"),
        col("tip_amount").cast("double").alias("tip_amount"),
        col("tolls_amount").cast("double").alias("tolls_amount"),
        col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
        col("total_amount").cast("double").alias("total_amount"),
        col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
        col(airport_column).cast("double").alias("airport_fee")
    )

In [17]:
jan_raw_df = (
    spark.read
    .schema(jan_source_schema)
    .parquet("data/yellow_tripdata_2023-01.parquet")
)

jan_df = standardise_trip_df(
    jan_raw_df,
    "airport_fee"
)

In [18]:
monthly_dfs = [jan_df]

for month in range(2, 13):
    path = f"data/yellow_tripdata_2023-{month:02d}.parquet"

    month_raw_df = (
        spark.read
        .schema(later_source_schema)
        .parquet(path)
    )

    month_df = standardise_trip_df(
        month_raw_df,
        "Airport_fee"
    )

    monthly_dfs.append(month_df)

In [19]:
from functools import reduce

trips_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    monthly_dfs
)

In [20]:
trips_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [21]:
trips_df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "airport_fee"
).show(5, truncate=False)

+--------+--------------------+---------------------+---------------+----------+------------+------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|RatecodeID|PULocationID|DOLocationID|airport_fee|
+--------+--------------------+---------------------+---------------+----------+------------+------------+-----------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |1              |1         |161         |141         |0.0        |
|2       |2023-01-01 00:55:08 |2023-01-01 01:01:27  |1              |1         |43          |237         |0.0        |
|2       |2023-01-01 00:25:04 |2023-01-01 00:37:49  |1              |1         |48          |238         |0.0        |
|1       |2023-01-01 00:03:48 |2023-01-01 00:13:25  |0              |1         |138         |7           |1.25       |
|2       |2023-01-01 00:10:29 |2023-01-01 00:21:19  |1              |1         |107         |79          |0.0        |
+--------+--------------------+-----------------

In [22]:
print("Number of monthly DataFrames:", len(monthly_dfs))
print("Total trip records:", trips_df.count())

Number of monthly DataFrames: 12
Total trip records: 38310226


In [23]:
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

In [24]:
from pyspark.sql.functions import (
    col,
    to_date,
    hour,
    dayofweek,
    unix_timestamp
)

trips_derived_df = (
    trips_df
    .withColumn(
        "pickup_date",
        to_date(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "pickup_hour",
        hour(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "day_of_week",
        dayofweek(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "trip_duration_minutes",
        (
            unix_timestamp(col("tpep_dropoff_datetime"))
            - unix_timestamp(col("tpep_pickup_datetime"))
        ) / 60.0
    )
    .withColumn(
        "pickup_epoch_seconds",
        unix_timestamp(col("tpep_pickup_datetime")).cast("long")
    )
)

In [25]:
trips_derived_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_date",
    "pickup_hour",
    "day_of_week",
    "trip_duration_minutes",
    "pickup_epoch_seconds"
).show(10, truncate=False)

+--------------------+---------------------+-----------+-----------+-----------+---------------------+--------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_date|pickup_hour|day_of_week|trip_duration_minutes|pickup_epoch_seconds|
+--------------------+---------------------+-----------+-----------+-----------+---------------------+--------------------+
|2023-01-01 00:32:10 |2023-01-01 00:40:36  |2023-01-01 |0          |1          |8.433333333333334    |1672551130          |
|2023-01-01 00:55:08 |2023-01-01 01:01:27  |2023-01-01 |0          |1          |6.316666666666666    |1672552508          |
|2023-01-01 00:25:04 |2023-01-01 00:37:49  |2023-01-01 |0          |1          |12.75                |1672550704          |
|2023-01-01 00:03:48 |2023-01-01 00:13:25  |2023-01-01 |0          |1          |9.616666666666667    |1672549428          |
|2023-01-01 00:10:29 |2023-01-01 00:21:19  |2023-01-01 |0          |1          |10.833333333333334   |1672549829          |
|2023-01

In [26]:
print("Taxi zone records:", zone_df.count())
zone_df.show(5, truncate=False)

Taxi zone records: 265
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


In [27]:
pickup_zone_df = zone_df.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("pickup_borough"),
    col("Zone").alias("pickup_zone"),
    col("service_zone").alias("pickup_service_zone")
)

dropoff_zone_df = zone_df.select(
    col("LocationID").alias("DOLocationID"),
    col("Borough").alias("dropoff_borough"),
    col("Zone").alias("dropoff_zone"),
    col("service_zone").alias("dropoff_service_zone")
)

In [28]:
trips_joined_df = (
    trips_derived_df
    .join(
        pickup_zone_df,
        on="PULocationID",
        how="left"
    )
    .join(
        dropoff_zone_df,
        on="DOLocationID",
        how="left"
    )
)

In [29]:
trips_joined_df.select(
    "PULocationID",
    "pickup_borough",
    "pickup_zone",
    "DOLocationID",
    "dropoff_borough",
    "dropoff_zone"
).show(10, truncate=False)

+------------+--------------+---------------------+------------+---------------+-----------------------------------+
|PULocationID|pickup_borough|pickup_zone          |DOLocationID|dropoff_borough|dropoff_zone                       |
+------------+--------------+---------------------+------------+---------------+-----------------------------------+
|161         |Manhattan     |Midtown Center       |141         |Manhattan      |Lenox Hill West                    |
|43          |Manhattan     |Central Park         |237         |Manhattan      |Upper East Side South              |
|48          |Manhattan     |Clinton East         |238         |Manhattan      |Upper West Side North              |
|138         |Queens        |LaGuardia Airport    |7           |Queens         |Astoria                            |
|107         |Manhattan     |Gramercy             |79          |Manhattan      |East Village                       |
|161         |Manhattan     |Midtown Center       |137         |

In [30]:
print("Records before joins:", trips_derived_df.count())
print("Records after joins:", trips_joined_df.count())

Records before joins: 38310226
Records after joins: 38310226


In [31]:
from pyspark.sql.functions import col, sum as spark_sum

trips_joined_df.select(
    spark_sum(
        col("pickup_borough").isNull().cast("int")
    ).alias("unmatched_pickup_locations"),

    spark_sum(
        col("dropoff_borough").isNull().cast("int")
    ).alias("unmatched_dropoff_locations")
).show()

+--------------------------+---------------------------+
|unmatched_pickup_locations|unmatched_dropoff_locations|
+--------------------------+---------------------------+
|                         0|                          0|
+--------------------------+---------------------------+



In [32]:
trips_joined_df.filter(
    col("pickup_borough").isNull()
).groupBy(
    "PULocationID"
).count().orderBy(
    col("count").desc()
).show(20)

+------------+-----+
|PULocationID|count|
+------------+-----+
+------------+-----+



In [33]:
trips_joined_df.filter(
    col("dropoff_borough").isNull()
).groupBy(
    "DOLocationID"
).count().orderBy(
    col("count").desc()
).show(20)

+------------+-----+
|DOLocationID|count|
+------------+-----+
+------------+-----+



In [34]:
from pyspark.sql.functions import col, sum as spark_sum

trips_joined_df.select(
    spark_sum((col("trip_distance") <= 0).cast("int")).alias("non_positive_distance"),
    spark_sum((col("fare_amount") < 0).cast("int")).alias("negative_fare"),
    spark_sum((col("trip_duration_minutes") <= 0).cast("int")).alias("non_positive_duration")
).show()

+---------------------+-------------+---------------------+
|non_positive_distance|negative_fare|non_positive_duration|
+---------------------+-------------+---------------------+
|               773457|       381650|                15569|
+---------------------+-------------+---------------------+



In [35]:
from pyspark.sql.functions import col

invalid_df = trips_joined_df.filter(
    (col("trip_distance") <= 0) |
    (col("fare_amount") < 0) |
    (col("trip_duration_minutes") <= 0)
)

print("Records failing at least one validity check:", invalid_df.count())

Records failing at least one validity check: 1117974


In [36]:
trips_joined_df.filter(
    (col("pickup_date") < "2023-01-01") |
    (col("pickup_date") > "2023-12-31")
).select(
    "pickup_date"
).groupBy(
    "pickup_date"
).count().orderBy(
    "pickup_date"
).show(50)

+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2001-01-01|    6|
| 2002-12-31|   11|
| 2003-01-01|    6|
| 2008-12-31|   23|
| 2009-01-01|   15|
| 2014-11-19|    1|
| 2022-10-24|    4|
| 2022-10-25|    7|
| 2022-12-31|   25|
| 2024-01-01|    2|
| 2024-01-03|    4|
+-----------+-----+



In [37]:
valid_trips_df = trips_joined_df.filter(
    (col("pickup_date") >= "2023-01-01") &
    (col("pickup_date") <= "2023-12-31") &
    (col("trip_distance") > 0) &
    (col("fare_amount") >= 0) &
    (col("trip_duration_minutes") > 0)
)

In [38]:
total_records = trips_joined_df.count()
valid_records = valid_trips_df.count()

print("Total records:", total_records)
print("Valid records:", valid_records)
print("Records excluded:", total_records - valid_records)

Total records: 38310226
Valid records: 37192160
Records excluded: 1118066


In [39]:
analysis_df = valid_trips_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "pickup_zone",
    "dropoff_zone",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount"
)

In [40]:
from pyspark.sql.functions import col, count, sum as spark_sum

MIN_TRIPS = 100

route_agg_df = (
    analysis_df
    .groupBy(
        "pickup_borough",
        "day_of_week",
        "pickup_hour",
        "PULocationID",
        "DOLocationID",
        "pickup_zone",
        "dropoff_zone"
    )
    .agg(
        count("*").alias("trip_count"),
        spark_sum("trip_distance").alias("total_distance"),
        spark_sum("trip_duration_minutes").alias("total_duration_minutes"),
        spark_sum("fare_amount").alias("total_fare")
    )
    .filter(col("trip_count") >= MIN_TRIPS)
    .withColumn(
        "route_speed_mph",
        col("total_distance") /
        (col("total_duration_minutes") / 60.0)
    )
    .withColumn(
        "fare_per_occupied_minute",
        col("total_fare") /
        col("total_duration_minutes")
    )
)

In [41]:
route_agg_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    "route_speed_mph"
).limit(5).show(truncate=True)

+--------------+-----------+-----------+--------------------+-------------------+----------+------------------+
|pickup_borough|day_of_week|pickup_hour|         pickup_zone|       dropoff_zone|trip_count|   route_speed_mph|
+--------------+-----------+-----------+--------------------+-------------------+----------+------------------+
|        Queens|          1|          0|         JFK Airport|Crown Heights North|       125|22.501573923853265|
|     Manhattan|          1|          0|Sutton Place/Turt...|UN/Turtle Bay South|       112|11.684137291280148|
|     Manhattan|          1|          0|Upper West Side S...|     Yorkville West|       221|11.544066852367687|
|     Manhattan|          1|          0|Meatpacking/West ...|           Flatiron|       210| 9.726435661340375|
|     Manhattan|          1|          0|            Flatiron|           Flatiron|       103| 9.378241430700445|
+--------------+-----------+-----------+--------------------+-------------------+----------+------------

In [42]:
benchmark_df = (
    analysis_df
    .groupBy(
        "pickup_borough",
        "day_of_week",
        "pickup_hour"
    )
    .agg(
        spark_sum("trip_distance").alias("benchmark_total_distance"),
        spark_sum("trip_duration_minutes").alias("benchmark_total_duration"),
        spark_sum("fare_amount").alias("benchmark_total_fare")
    )
    .withColumn(
        "benchmark_speed_mph",
        col("benchmark_total_distance") /
        (col("benchmark_total_duration") / 60.0)
    )
    .withColumn(
        "benchmark_fare_per_minute",
        col("benchmark_total_fare") /
        col("benchmark_total_duration")
    )
)

In [43]:
benchmark_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "benchmark_speed_mph",
    "benchmark_fare_per_minute"
).show(20, truncate=False)

+--------------+-----------+-----------+-------------------+-------------------------+
|pickup_borough|day_of_week|pickup_hour|benchmark_speed_mph|benchmark_fare_per_minute|
+--------------+-----------+-----------+-------------------+-------------------------+
|N/A           |2          |3          |21.117848668600054 |12.61376219351437        |
|Unknown       |7          |21         |12.064803633525962 |1.077850739183751        |
|N/A           |5          |15         |12.737447683177088 |1.1574086905043413       |
|Bronx         |1          |10         |14.832797222482881 |1.0597136154640145       |
|Manhattan     |1          |15         |13.33831395083233  |1.0482563197683439       |
|N/A           |3          |13         |15.0239633817986   |1.522513911326512        |
|N/A           |5          |13         |15.8273698490758   |2.04542705330394         |
|Staten Island |1          |10         |25.338873386280707 |1.2120143509858636       |
|Bronx         |4          |2          |23.

In [44]:
route_comparison_df = (
    route_agg_df
    .join(
        benchmark_df,
        on=[
            "pickup_borough",
            "day_of_week",
            "pickup_hour"
        ],
        how="inner"
    )
    .withColumn(
        "speed_deficit_percentage",
        (
            (
                col("benchmark_speed_mph") -
                col("route_speed_mph")
            )
            / col("benchmark_speed_mph")
        ) * 100
    )
)

In [45]:
route_comparison_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    "route_speed_mph",
    "benchmark_speed_mph",
    "speed_deficit_percentage",
    "fare_per_occupied_minute",
    "benchmark_fare_per_minute"
).limit(5).show(truncate=True)

+--------------+-----------+-----------+--------------------+-------------------+----------+------------------+-------------------+------------------------+------------------------+-------------------------+
|pickup_borough|day_of_week|pickup_hour|         pickup_zone|       dropoff_zone|trip_count|   route_speed_mph|benchmark_speed_mph|speed_deficit_percentage|fare_per_occupied_minute|benchmark_fare_per_minute|
+--------------+-----------+-----------+--------------------+-------------------+----------+------------------+-------------------+------------------------+------------------------+-------------------------+
|        Queens|          1|          0|         JFK Airport|Crown Heights North|       125|22.501573923853265|  28.50194000848769|      21.052483033953322|       1.568159120119677|       1.9006099023339222|
|     Manhattan|          1|          0|Sutton Place/Turt...|UN/Turtle Bay South|       112|11.684137291280148| 11.439348008355672|      -2.139888416242549|      1.5925

In [46]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

route_window = (
    Window
    .partitionBy(
        "pickup_borough",
        "day_of_week",
        "pickup_hour"
    )
    .orderBy(
        col("speed_deficit_percentage").desc()
    )
)

ranked_routes_df = (
    route_comparison_df
    .withColumn(
        "route_rank",
        row_number().over(route_window)
    )
    .filter(
        col("route_rank") <= 3
    )
)

In [47]:
ranked_routes_df.filter(
    (col("pickup_borough") == "Manhattan") &
    (col("day_of_week") == 1) &
    (col("pickup_hour") == 15)
).select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    "route_speed_mph",
    "benchmark_speed_mph",
    "speed_deficit_percentage",
    "fare_per_occupied_minute",
    "benchmark_fare_per_minute",
    "route_rank"
).orderBy("route_rank").show(truncate=False)

+--------------+-----------+-----------+----------------+----------------+----------+------------------+-------------------+------------------------+------------------------+-------------------------+----------+
|pickup_borough|day_of_week|pickup_hour|pickup_zone     |dropoff_zone    |trip_count|route_speed_mph   |benchmark_speed_mph|speed_deficit_percentage|fare_per_occupied_minute|benchmark_fare_per_minute|route_rank|
+--------------+-----------+-----------+----------------+----------------+----------+------------------+-------------------+------------------------+------------------------+-------------------------+----------+
|Manhattan     |1          |15         |Midtown South   |Midtown South   |119       |1.4492263617710446|13.33831395083233  |89.13486091935472       |0.501352415119175       |1.0482563197683439       |1         |
|Manhattan     |1          |15         |Garment District|Murray Hill     |121       |1.6315032516258126|13.33831395083233  |87.76829472120798       |0.3

In [48]:
ranked_routes_df = (
    ranked_routes_df
    .withColumn(
        "lower_fare_than_benchmark",
        col("fare_per_occupied_minute") <
        col("benchmark_fare_per_minute")
    )
)

In [49]:
ranked_routes_df.filter(
    (col("pickup_borough") == "Manhattan") &
    (col("day_of_week") == 1) &
    (col("pickup_hour") == 15)
).select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    "route_speed_mph",
    "benchmark_speed_mph",
    "speed_deficit_percentage",
    "fare_per_occupied_minute",
    "benchmark_fare_per_minute",
    "lower_fare_than_benchmark",
    "route_rank"
).orderBy("route_rank").show(truncate=False)

+--------------+-----------+-----------+----------------+----------------+----------+------------------+-------------------+------------------------+------------------------+-------------------------+-------------------------+----------+
|pickup_borough|day_of_week|pickup_hour|pickup_zone     |dropoff_zone    |trip_count|route_speed_mph   |benchmark_speed_mph|speed_deficit_percentage|fare_per_occupied_minute|benchmark_fare_per_minute|lower_fare_than_benchmark|route_rank|
+--------------+-----------+-----------+----------------+----------------+----------+------------------+-------------------+------------------------+------------------------+-------------------------+-------------------------+----------+
|Manhattan     |1          |15         |Midtown South   |Midtown South   |119       |1.4492263617710446|13.33831395083233  |89.13486091935472       |0.501352415119175       |1.0482563197683439       |true                     |1         |
|Manhattan     |1          |15         |Garment 

In [50]:
analysis_df.createOrReplaceTempView("valid_trips")

In [51]:
sql_result_df = spark.sql("""
WITH route_agg AS (
    SELECT
        pickup_borough,
        day_of_week,
        pickup_hour,
        PULocationID,
        DOLocationID,
        pickup_zone,
        dropoff_zone,
        COUNT(*) AS trip_count,
        SUM(trip_distance) AS total_distance,
        SUM(trip_duration_minutes) AS total_duration_minutes,
        SUM(fare_amount) AS total_fare
    FROM valid_trips
    GROUP BY
        pickup_borough,
        day_of_week,
        pickup_hour,
        PULocationID,
        DOLocationID,
        pickup_zone,
        dropoff_zone
    HAVING COUNT(*) >= 100
),

route_metrics AS (
    SELECT
        *,
        total_distance / (total_duration_minutes / 60.0)
            AS route_speed_mph,
        total_fare / total_duration_minutes
            AS fare_per_occupied_minute
    FROM route_agg
),

benchmark AS (
    SELECT
        pickup_borough,
        day_of_week,
        pickup_hour,
        SUM(trip_distance) / (SUM(trip_duration_minutes) / 60.0)
            AS benchmark_speed_mph,
        SUM(fare_amount) / SUM(trip_duration_minutes)
            AS benchmark_fare_per_minute
    FROM valid_trips
    GROUP BY
        pickup_borough,
        day_of_week,
        pickup_hour
),

comparison AS (
    SELECT
        r.*,
        b.benchmark_speed_mph,
        b.benchmark_fare_per_minute,
        (
            (b.benchmark_speed_mph - r.route_speed_mph)
            / b.benchmark_speed_mph
        ) * 100 AS speed_deficit_percentage
    FROM route_metrics r
    INNER JOIN benchmark b
        ON r.pickup_borough = b.pickup_borough
        AND r.day_of_week = b.day_of_week
        AND r.pickup_hour = b.pickup_hour
),

ranked AS (
    SELECT
        *,
        fare_per_occupied_minute < benchmark_fare_per_minute
            AS lower_fare_than_benchmark,
        ROW_NUMBER() OVER (
            PARTITION BY
                pickup_borough,
                day_of_week,
                pickup_hour
            ORDER BY speed_deficit_percentage DESC
        ) AS route_rank
    FROM comparison
)

SELECT *
FROM ranked
WHERE route_rank <= 3
""")

In [52]:
sql_result_df.filter(
    (col("pickup_borough") == "Manhattan") &
    (col("day_of_week") == 1) &
    (col("pickup_hour") == 15)
).select(
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    "speed_deficit_percentage",
    "lower_fare_than_benchmark",
    "route_rank"
).orderBy("route_rank").show(truncate=False)

+----------------+----------------+----------+------------------------+-------------------------+----------+
|pickup_zone     |dropoff_zone    |trip_count|speed_deficit_percentage|lower_fare_than_benchmark|route_rank|
+----------------+----------------+----------+------------------------+-------------------------+----------+
|Midtown South   |Midtown South   |119       |89.13486091935472       |true                     |1         |
|Garment District|Murray Hill     |121       |87.76829472120798       |true                     |2         |
|Clinton East    |Garment District|119       |83.82491136622801       |true                     |3         |
+----------------+----------------+----------+------------------------+-------------------------+----------+



In [53]:
df_validation = ranked_routes_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "trip_count",
    "route_rank"
)

sql_validation = sql_result_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "trip_count",
    "route_rank"
)

print("DataFrame result rows:", df_validation.count())
print("SQL result rows:", sql_validation.count())

print(
    "DataFrame rows not in SQL:",
    df_validation.exceptAll(sql_validation).count()
)

print(
    "SQL rows not in DataFrame:",
    sql_validation.exceptAll(df_validation).count()
)

DataFrame result rows: 1093
SQL result rows: 1093
DataFrame rows not in SQL: 0
SQL rows not in DataFrame: 0


The DataFrame and Spark SQL implementations produced the same 1,093 result rows, with zero unmatched rows in either direction. The DataFrame implementation expresses the query through chained transformations, while the SQL implementation separates the route aggregation, benchmark calculation, comparison and ranking into CTEs. The SQL structure makes the individual query stages explicit, while the DataFrame approach remains consistent with the earlier Spark preprocessing steps. Both approaches implement the same analytical logic and produce equivalent results.